# Actividad 4: Aplicación de algoritmos de aprendizaje no supervisado con PySpark

**Materia:** Análisis de grandes volúmenes de datos  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Autor:** Jonathan Javier Monsalve Giraldo (A01840272)  
**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 1 de junio de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025  
**Modalidad:** Individual


## Objetivo

Aplicar algoritmos de aprendizaje no supervisado en PySpark MLlib sobre una muestra M' derivada de la muestra estratificada M construida en la Etapa 2 del proyecto del equipo. El problema elegido es la segmentación de viajes Yellow Taxi NYC en arquetipos operativos, sin utilizar una variable objetivo, para descubrir perfiles naturales de viaje según distancia, duración, velocidad, zona de origen, horario, pasajeros y régimen Flex Fare.

## Estructura del notebook

1. **Introducción**: aprendizaje no supervisado, algoritmos representativos y los disponibles en PySpark MLlib.
2. **Selección de los datos**: reconstrucción compacta de M, construcción de la muestra individual M' y derivación de variables para clustering.
3. **Preparación del conjunto de entrenamiento y prueba**: partición estratificada train/test y validación del split como revisión de estabilidad.
4. **Construcción de modelos de aprendizaje no supervisado**: selección de features, pipeline, barrido de k, KMeans, comparación con GMM, evaluación e interpretación de arquetipos.

### Nota para el profesor

Las secciones 2.0, 2.1 y 2.2 reutilizan de forma compacta la reconstrucción de M desde la Etapa 2, la construcción de M' y la validación de representatividad usadas en la Actividad 3. Se conservan para que el notebook sea autocontenido y reproducible. El aporte nuevo de esta actividad inicia en la **Sección 2.3**, donde se derivan las variables operativas para clustering; continúa en la **Sección 3**, donde el split se usa para medir estabilidad del patrón, y en la **Sección 4**, donde se entrenan e interpretan los modelos no supervisados.


## 1. Introducción

### 1.1 Aprendizaje no supervisado

El aprendizaje no supervisado agrupa métodos que buscan estructura interna en datos sin una variable objetivo conocida. A diferencia del aprendizaje supervisado, donde cada observación incluye una etiqueta para entrenar y evaluar predicciones, aquí el modelo identifica similitudes, patrones, componentes latentes o casos poco usuales a partir de las variables disponibles.

La calidad de un resultado no supervisado no se mide comparando contra una respuesta verdadera, sino con criterios internos y con interpretación del dominio. Por ello, métricas como cohesión, separación, varianza explicada, probabilidad de pertenencia o frecuencia de patrones se usan como guías, pero deben complementarse con una lectura razonada de los grupos o estructuras descubiertas.

### 1.2 Familias representativas

Las técnicas no supervisadas se organizan en varias familias. El **clustering** agrupa observaciones similares; KMeans representa cada grupo por un centroide, BisectingKMeans construye divisiones jerárquicas y Gaussian Mixture Model permite pertenencia probabilística a componentes gaussianos. La **reducción de dimensión**, como PCA o SVD, resume muchas variables en menos componentes que conservan variabilidad. Las **reglas de asociación**, como FPGrowth, descubren combinaciones frecuentes de ítems o eventos. En **texto**, LDA identifica temas latentes a partir de vectores de conteos. La **detección de anomalías** puede aproximarse con distancia al centroide, baja probabilidad de pertenencia o reglas de negocio cuando no existe una etiqueta de fraude o error.

### 1.3 Algoritmos disponibles en PySpark MLlib

PySpark expone estas técnicas mediante la API moderna `pyspark.ml`, basada en DataFrames y en el patrón `Estimator`/`Transformer`/`Pipeline`. Un `Estimator` aprende parámetros con `fit()`, un `Transformer` agrega columnas con `transform()`, y un `Pipeline` encadena preparación de datos y modelo para aplicar en test las transformaciones aprendidas en train.

| Familia | Submódulo PySpark | Implementaciones relevantes | Uso típico |
|---|---|---|---|
| Clustering | `pyspark.ml.clustering` | `KMeans`, `BisectingKMeans`, `GaussianMixture`, `LDA`, `PowerIterationClustering` | Segmentación tabular, temas en texto o clusters sobre grafos |
| Reducción de dimensión | `pyspark.ml.feature` | `PCA` | Compresión de variables, visualización o preprocesamiento antes de clustering |
| Patrones frecuentes | `pyspark.ml.fpm` | `FPGrowth`, `PrefixSpan` | Canastas, secuencias y combinaciones frecuentes de eventos |
| Evaluación | `pyspark.ml.evaluation` | `ClusteringEvaluator` | Cálculo de silhouette como métrica interna de cohesión y separación |

La tabla resume las clases de alto nivel disponibles en la API DataFrame `pyspark.ml`, no las clases auxiliares de modelo, resumen o la API RDD antigua `pyspark.mllib`. La documentación oficial de Spark lista en clustering `KMeans`, `LDA`, `BisectingKMeans`, `GaussianMixture` y `PowerIterationClustering`; en patrones frecuentes, `FPGrowth` y `PrefixSpan`; y en transformación de features, `PCA` para reducción de dimensión. KMeans, BisectingKMeans y GMM producen una columna de cluster y pueden evaluarse con `ClusteringEvaluator` mediante silhouette. En no supervisado esta métrica no equivale a una verdad externa: se usa como criterio interno junto con tamaño de clusters, estabilidad train/test e interpretación de los perfiles.

### 1.4 Referencias

Apache Software Foundation. (2026). *Clustering - Spark 4.1.2 Documentation*. https://spark.apache.org/docs/latest/ml-clustering.html

Apache Software Foundation. (2026). *Extracting, transforming and selecting features - Spark 4.1.2 Documentation*. https://spark.apache.org/docs/latest/ml-features.html

Apache Software Foundation. (2026). *Frequent Pattern Mining - Spark 4.1.2 Documentation*. https://spark.apache.org/docs/latest/ml-frequent-pattern-mining.html

Polak, A. (2023). *Scaling machine learning with Spark: Distributed ML with MLlib, TensorFlow, and PyTorch*. O'Reilly Media. Chapter 6: Unsupervised Machine Learning.
